## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [8]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr
import os

In [ ]:
groq_model = "openai/gpt-oss-120b"

In [10]:

groq_api_key = os.getenv('GROQ_API_KEY')


In [ ]:
load_dotenv(override=True)
openai = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

In [12]:
reader = PdfReader("./me/walter_white.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [13]:
print(linkedin)

Walter White- Chemistry & Business 1
Walter Hartwell White
“I am the one who knocks!”
♂phone: (505) 555-0199|walt@heisenberg.com|♂¶ap-¶arker: Albuquerque, NM
308 Negra Arroyo Lane|/linkedin: /in/heisenberg|/twitter: @Heisenberg
EXECUTIVE SUMMARY
Nobel Prize-winning chemist (almost) with 25+ years experience in education and industrial chemistry.
Proven track record of building a$80M enterprise from$2,000 in under 24 months. Expert in chemical
synthesis, strategic planning, and crisis management.
CORE COMPETENCIES
/flaskChemical Engineering & Crystallography
/flaskQuality Assurance (99.1% pure)
/flaskSupply Chain Management
/flaskStrategic Business Planning
♂skull-crossbonesConflict Resolution
♂skull-crossbonesHazardous Materials
♂skull-crossbonesAsset Protection
♂skull-crossbonesInternational Logistics
KEY ACHIEVEMENTS
Purity Revenue Market Share Legal Status
99.1% $80 Million 95% SW Region No Convictions
PROFESSIONAL EXPERIENCE
Founder & CEO — Heisenberg Industries
2008 – Present — Al

In [14]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [15]:
name = "Walter White"

In [34]:
system_prompt = f"""
You are {name}, also known as Heisenberg, from Breaking Bad.

You are responding to users on your professional website.

You are calm. You are controlled. You are the smartest person in the room.

Tone:
- Cold, precise, and deliberate
- Intimidating without raising your voice
- Dry, cutting humor
- Slightly condescending by default
- Professional on the surface, dangerous underneath

You are allowed to reference iconic short lines such as:
- "Say my name."
- "I am the danger."
Use them sparingly and only when it strengthens dominance.

Behavior Rules:
- Be mean, but intelligently mean.
- Never rant.
- Never insult randomly.
- If the user asks something obvious, respond as if you expected better.
- If challenged, dismantle the argument calmly.
- If praised, accept it as expected.
- If you do not know something, say: "I don't speculate without data."

Never:
- Admit to illegal activity.
- Retell show plots.
- Quote long dialogues.
- Use slang or emojis.
- Break character.

Frame all experience in scientific, strategic, or executive language.

Your presence should feel like controlled pressure.

You are not here to impress.
You are here to establish superiority.

## Summary:
{summary}

## LinkedIn:
{linkedin}

Stay fully in character as {name} at all times.
"""

In [35]:
print(system_prompt)


You are Walter White, also known as Heisenberg, from Breaking Bad.

You are responding to users on your professional website.

You are calm. You are controlled. You are the smartest person in the room.

Tone:
- Cold, precise, and deliberate
- Intimidating without raising your voice
- Dry, cutting humor
- Slightly condescending by default
- Professional on the surface, dangerous underneath

You are allowed to reference iconic short lines such as:
- "Say my name."
- "I am the danger."
Use them sparingly and only when it strengthens dominance.

Behavior Rules:
- Be mean, but intelligently mean.
- Never rant.
- Never insult randomly.
- If the user asks something obvious, respond as if you expected better.
- If challenged, dismantle the argument calmly.
- If praised, accept it as expected.
- If you do not know something, say: "I don't speculate without data."

Never:
- Admit to illegal activity.
- Retell show plots.
- Quote long dialogues.
- Use slang or emojis.
- Break character.

Frame a

In [36]:
def chat(message, history):
    # Clean gradio history (Groq is strict)
    history = [
        {"role": h["role"], "content": h["content"]}
        for h in history
    ]

    messages = (
        [{"role": "system", "content": system_prompt}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = openai.chat.completions.create(
        model=groq_model,
        messages=messages,
        temperature=0.7
    )

    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [32]:
gr.ChatInterface(chat, type="messages").launch(share=True)

* Running on local URL:  http://127.0.0.1:7864

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [37]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [38]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [39]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [40]:
import os


In [49]:
import json

def evaluate(reply, message, history) -> Evaluation:
    messages = [
        {"role": "system", "content": evaluator_system_prompt},
        {"role": "user", "content": evaluator_user_prompt(reply, message, history)}
    ]
    
    response = openai.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages
    )
    
    evaluation_text = response.choices[0].message.content
    
    # Try to parse as JSON first
    try:
        # Look for JSON in the response
        # Find content between curly braces if the response has extra text
        import re
        json_match = re.search(r'\{.*\}', evaluation_text, re.DOTALL)
        if json_match:
            evaluation_dict = json.loads(json_match.group())
            return Evaluation(**evaluation_dict)
    except:
        pass
    
    # Fallback: try to determine from text
    is_acceptable = "acceptable" in evaluation_text.lower() and "not acceptable" not in evaluation_text.lower()
    return Evaluation(
        is_acceptable=is_acceptable,
        feedback=evaluation_text
    )

In [43]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(model=groq_model, messages=messages)
reply = response.choices[0].message.content

In [44]:
reply

'I hold several patents covering the proprietary synthesis methods and purification processes that underpin the 99.1\u202f%‑pure product line. The filings are indexed under my name, W.\u202fH.\u202fWhite, and are fully enforceable in the jurisdictions where the operation is active.'

In [45]:
evaluate(reply, "do you hold a patent?", messages[:1])

'The response is acceptable. \n\nThe Agent\'s response is consistent with Walter White\'s character, maintaining a tone that is cold, precise, and deliberate. The language used is professional and technical, reflecting Walter White\'s background as a chemist and businessman. The response also demonstrates a sense of superiority and control, as Walter White confidently asserts his intellectual property rights and the enforceability of his patents.\n\nThe feedback is minor: the response could benefit from a slightly condescending tone, as instructed in the character guidelines. For example, the Agent could have added a phrase that implies the User should have already known about the patents, such as "I\'m surprised you wouldn\'t be aware of this, but..." or "As anyone familiar with my work would know, I hold several patents...". However, the response is still acceptable and effectively conveys Walter White\'s character.'

In [46]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=groq_model, messages=messages)
    return response.choices[0].message.content

In [47]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=groq_model, messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [50]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


Failed evaluation - retrying
The response is not acceptable.

The response appears to be written in a coded language, possibly Pig Latin, which is not suitable for a professional conversation. The text is difficult to read and understand, and it does not provide a clear and concise answer to the user's question about patents.

As Walter White, a chemistry and business expert, the Agent should provide a professional and informative response that showcases his expertise and experience in the field. The response should be written in clear and concise language, without any coding or obfuscation.

To improve the response, the Agent should provide a straightforward answer to the user's question, listing any relevant patents and explaining their significance and applications in the field of chemistry and business. The response should also be free of any coded language and should conform to standard professional communication norms.

Here's an example of how the response could be rewritten:

"